In [ ]:
from env import config, BASE_DIR
from sqlalchemy import create_engine

In [ ]:
import pandas as pd
import datetime

In [ ]:
DATABASE_URL = config("FLIGHT_PRICES_DATABASE_URL", cast=str, default=None)
DATABASE_URL is not None

In [ ]:
engine = create_engine(str(DATABASE_URL))

In [ ]:
DATASET_DIR = BASE_DIR / "datasets"
DATASET_DIR.exists()

In [ ]:
GRETEL_DIR = DATASET_DIR / "gretel"
GRETEL_DIR.exists()

In [ ]:
table_name = "flight_prices"
df = pd.read_sql(table_name, engine, index_col="index")
df.tail(n=1)
# df = out_df.to_sql(table_name, engine, if_exists='replace', index=True)

In [ ]:
last_row = df.tail(n=1)
last_idx = last_row.index[0]
last_idx

In [ ]:
max_flight_date = df.flightDate.max()
max_flight_date

In [ ]:
import glob
import os

gretel_dfs = []
csvfiles = glob.glob(os.path.join(GRETEL_DIR, "*.csv"))

for csvfile in csvfiles:
    df = pd.read_csv(csvfile)
    gretel_dfs.append(df)

new_df = pd.concat(gretel_dfs)
new_df.index = new_df.index + last_idx + 1
new_df["flightDate"] = pd.to_datetime(new_df["flightDate"])
new_df["flightDate"] = new_df["flightDate"].apply(
    lambda x: max_flight_date + (max_flight_date - x) + datetime.timedelta(days=1)
)
new_df.head()

In [ ]:
new_df.flightDate.max().date()

In [ ]:
new_df.to_sql(table_name, engine, if_exists="append", index=True)

In [ ]:
df = pd.read_sql(table_name, engine, index_col="index")
df.tail()